# Setup

In [ ]:
import sys
from pathlib import Path
import os
import importlib

os.environ["MX_DEX_ENV"] = "shadowfork4"

sys.path.append(str(Path.cwd().parent.parent.parent.absolute()))
import config
import argparse
importlib.reload(config)
from context import Context
from utils.utils_chain import WrapperAddress
from contracts.router_contract import RouterContract
from contracts.pair_contract import PairContract
from contracts.farm_contract import FarmContract
from contracts.staking_contract import StakingContract
from contracts.governance_contract import GovernanceContract
from tools.runners.farm_runner import upgrade_farmv2_contracts, get_all_farm_v2_addresses, fetch_and_save_farms_from_chain,\
    pause_farm_contracts, resume_farm_contracts
from tools.runners.staking_runner import upgrade_staking_contracts, get_staking_addresses_from_chain,\
    pause_all_staking_contracts, resume_all_staking_contracts, fetch_and_save_stakings_from_chain
from tools.runners.pair_runner import upgrade_pair_contracts, get_all_pair_addresses, fetch_and_save_pairs_from_chain,\
    pause_pair_contracts, resume_pair_contracts
from tools.runners.router_runner import upgrade_router_contract, upgrade_template_pair_contract,\
    resume as resume_router, pause as pause_router
from tools.runner import fetch_and_save_pause_state
from utils.utils_chain import WrapperAddress, base64_to_hex

context = Context()

router_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.4.1-rc4/router.wasm"
router_code_hash = "4a7f6baf4aeebd1c9892b6bafd74ee3548b04be17d23fc0d554318015568d0a9"
pair_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.4.1-rc4/pair.wasm"
pair_code_hash = "4a7f6baf4aeebd1c9892b6bafd74ee3548b04be17d23fc0d554318015568d0a9"
pair_view_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.4.1-rc4/pair-view.wasm"
pair_view_code_hash = "4a7f6baf4aeebd1c9892b6bafd74ee3548b04be17d23fc0d554318015568d0a9"
farm_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.4.1-rc4/farm.wasm"
farm_code_hash = "4a7f6baf4aeebd1c9892b6bafd74ee3548b04be17d23fc0d554318015568d0a9"
staking_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.4.1-rc4/staking.wasm"
staking_code_hash = "4a7f6baf4aeebd1c9892b6bafd74ee3548b04be17d23fc0d554318015568d0a9"

governance_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.4.1-rc4/governance.wasm"
governance_code_hash = "4a7f6baf4aeebd1c9892b6bafd74ee3548b04be17d23fc0d554318015568d0a9"

Check owner balance for required minimum

In [ ]:
current_balance = context.network_provider.proxy.get_account(context.deployer_account.address).balance
assert current_balance > 20 * 10 ** 18, "Deployer account doesn't have enough balance"

Clean outputs folder

In [ ]:
import shutil
if config.UPGRADER_OUTPUT_FOLDER.exists():
    shutil.rmtree(config.UPGRADER_OUTPUT_FOLDER)

Prep contract pause states

In [ ]:
fetch_and_save_farms_from_chain("")

In [ ]:
fetch_and_save_stakings_from_chain("")

In [ ]:
fetch_and_save_pairs_from_chain("")

In [ ]:
fetch_and_save_pause_state("")

Pause contracts

In [ ]:
pause_farm_contracts("")

In [ ]:
pause_all_staking_contracts("")

In [ ]:
pause_router("")

In [ ]:
pause_pair_contracts("")

# Upgrade farm contracts

In [ ]:
args = argparse.Namespace(bytecode=farm_wasm, compare_states=True)
upgrade_farmv2_contracts(args)

farm_addresses = get_all_farm_v2_addresses()
print(f"Checking {len(farm_addresses)} farm contracts for correct code hash...")
for address in farm_addresses:
    code_hash = context.network_provider.proxy.get_account(WrapperAddress(address)).contract_code_hash.hex()
    assert code_hash == farm_code_hash, f"Code hash mismatch for {address}"
print("Done!")

# Upgrade staking contracts

In [ ]:
args = argparse.Namespace(bytecode=staking_wasm, compare_states=True, all=True)
upgrade_staking_contracts(args)

staking_addresses = get_staking_addresses_from_chain()
print(f"Checking {len(staking_addresses)} staking contracts for correct code hash...")
for address in staking_addresses:
    code_hash = context.network_provider.proxy.get_account(WrapperAddress(address)).contract_code_hash.hex()
    assert code_hash == staking_code_hash, f"Code hash mismatch for {address}"
print("Done!")

# Upgrade Safe Price

Upgrade router contract

In [ ]:
router_address = context.get_contracts(config.ROUTER_V2)[0].address
args = argparse.Namespace(address=router_address, bytecode=router_wasm, compare_states=True)
upgrade_router_contract(args)

code_hash = context.network_provider.proxy.get_account(WrapperAddress(router_address)).contract_code_hash.hex()
assert code_hash == router_code_hash, f"Code hash mismatch for {router_address}"
print("Done!")

Upgrade template pair contract

In [ ]:
args = argparse.Namespace(bytecode=pair_wasm, compare_states=True)
upgrade_template_pair_contract(args)

router_contract: RouterContract = context.get_contracts(config.ROUTER_V2)[0]
template_pair_address = router_contract.get_pair_template_address(context.network_provider.proxy)
code_hash = context.network_provider.proxy.get_account(WrapperAddress(template_pair_address)).contract_code_hash.hex()
assert code_hash == pair_code_hash, f"Code hash mismatch for {template_pair_address}"
print("Done!")

Upgrade view contract

Upgrade pairs

In [ ]:
args = argparse.Namespace(address=address, bytecode=pair_wasm, compare_states=True)
upgrade_pair_contracts(args)

code_hash = context.network_provider.proxy.get_account(WrapperAddress(address)).contract_code_hash.hex()
assert code_hash == pair_code_hash, f"Code hash mismatch for {address}"

# Upgrade Governance

# Resume contracts

In [ ]:
resume_router("")

In [ ]:
resume_pair_contracts("")

In [ ]:
resume_farm_contracts("")

In [ ]:
resume_all_staking_contracts("")